# Helper Models for the Streamlit App

The main price model needs `median_income` and `ocean_proximity`, which an ordinary app user
does not know. The app therefore estimates them from the clicked **latitude/longitude** with two
helper models trained here:

| Helper | Task | Candidates compared |
|---|---|---|
| Ocean proximity | Classification (5 classes) | k-NN with k chosen by 5-fold CV |
| Median income | Regression | k-NN regressor vs Random Forest vs XGBoost (GPU) |

**Run this notebook before `DMBI_Project_v2.ipynb`**. The main notebook uses these helpers to
measure the *real* error of the chained app (section 13 there).

Both notebooks use the same stratified split (`housing_features.split_data`), so the helpers
never see the main model's test rows.

In [1]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from housing_features import split_data, get_xgb_device

warnings.filterwarnings("ignore", category=UserWarning)
Path("models").mkdir(exist_ok=True)

DEVICE = get_xgb_device()
print("XGBoost device:", DEVICE)

data = pd.read_csv("housing.csv")
train_df, test_df = split_data(data)
LOC = ["latitude", "longitude"]
print(f"train={len(train_df):,} rows, test={len(test_df):,} rows")

XGBoost device: cuda
train=16,512 rows, test=4,128 rows


## 1. Ocean proximity: k-NN classifier

Ocean proximity is a purely geographic label, so the nearest census blocks on the map should
share it. Latitude and longitude are both in degrees, so no scaling is needed.

In [2]:
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn_search = GridSearchCV(
    KNeighborsClassifier(),
    {"n_neighbors": [1, 3, 5, 7, 9, 15], "weights": ["uniform", "distance"]},
    cv=cv_cls, scoring="accuracy", n_jobs=-1,
)
knn_search.fit(train_df[LOC], train_df["ocean_proximity"])
ocean_model = knn_search.best_estimator_

ocean_pred = ocean_model.predict(test_df[LOC])
ocean_acc = accuracy_score(test_df["ocean_proximity"], ocean_pred)
print("Best params:", knn_search.best_params_)
print(f"5-fold CV accuracy: {knn_search.best_score_:.4f}")
print(f"Test accuracy:      {ocean_acc:.4f}\n")
print(classification_report(test_df["ocean_proximity"], ocean_pred, zero_division=0))

Best params: {'n_neighbors': 3, 'weights': 'distance'}
5-fold CV accuracy: 0.9901
Test accuracy:      0.9918

              precision    recall  f1-score   support

   <1H OCEAN       0.99      0.99      0.99      1859
      INLAND       1.00      1.00      1.00      1289
      ISLAND       1.00      1.00      1.00         3
    NEAR BAY       0.99      0.99      0.99       443
  NEAR OCEAN       0.99      0.97      0.98       534

    accuracy                           0.99      4128
   macro avg       0.99      0.99      0.99      4128
weighted avg       0.99      0.99      0.99      4128



`ISLAND` has only 5 rows in the whole dataset, so its per-class score is not meaningful. The
other four classes are predicted almost perfectly.

## 2. Median income from location: comparing three regressors

Each candidate is scored with the same 5-fold CV on the training set. The winner is then checked
once on the held-out test set.

In [3]:
X_tr, y_tr = train_df[LOC], train_df["median_income"]
X_te, y_te = test_df[LOC], test_df["median_income"]
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    "k-NN regressor": KNeighborsRegressor(n_neighbors=10, weights="distance"),
    "Random Forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=42),
    f"XGBoost ({DEVICE})": XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=10, subsample=0.8,
                                        colsample_bytree=1.0, device=DEVICE, tree_method="hist", random_state=42),
}
scoring = {"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}

rows = []
for name, model in candidates.items():
    t0 = time.perf_counter()
    cv = cross_validate(model, X_tr, y_tr, cv=cv_reg, scoring=scoring, n_jobs=1)
    rows.append({
        "model": name,
        "CV R2": f"{cv['test_r2'].mean():.3f} ± {cv['test_r2'].std():.3f}",
        "CV MAE": f"{-cv['test_mae'].mean():.3f} ± {cv['test_mae'].std():.3f}",
        "CV RMSE": f"{-cv['test_rmse'].mean():.3f} ± {cv['test_rmse'].std():.3f}",
        "_rmse": -cv["test_rmse"].mean(),
        "seconds": round(time.perf_counter() - t0, 1),
    })
income_cv = pd.DataFrame(rows).sort_values("_rmse")
income_cv.drop(columns="_rmse")

,model,CV R2,CV MAE,CV RMSE,seconds
1,Random Forest,0.565 ± 0.011,0.871 ± 0.014,1.256 ± 0.028,13.1
0,k-NN regressor,0.505 ± 0.006,0.924 ± 0.010,1.340 ± 0.020,0.1
2,XGBoost (cuda),0.491 ± 0.007,0.934 ± 0.016,1.359 ± 0.028,42.7


In [4]:
best_name = income_cv.iloc[0]["model"]
income_model = candidates[best_name].fit(X_tr, y_tr)
inc_pred = income_model.predict(X_te)
income_test = {
    "model": best_name,
    "r2": r2_score(y_te, inc_pred),
    "mae": mean_absolute_error(y_te, inc_pred),
    "rmse": float(np.sqrt(mean_squared_error(y_te, inc_pred))),
}
print(f"Selected: {best_name}")
print(f"Test R² {income_test['r2']:.3f} | MAE {income_test['mae']:.3f} | RMSE {income_test['rmse']:.3f}  (units: $10k)")

Selected: Random Forest
Test R² 0.561 | MAE 0.865 | RMSE 1.245  (units: $10k)


**Interpretation.** Location alone explains only part of the variation in income, because
neighbouring blocks can differ a lot. The income helper is therefore the weakest link in the app,
and its error is carried into the price prediction. The main notebook measures how large that
effect is.

In [5]:
if hasattr(income_model, "set_params") and "device" in income_model.get_params():
    income_model.set_params(device="cpu")   # so the app also runs on machines without a GPU
joblib.dump(ocean_model, "models/ocean_model.joblib")
joblib.dump(income_model, "models/income_model.joblib")
with open("models/helper_metrics.json", "w") as f:
    json.dump({
        "ocean": {"params": knn_search.best_params_, "cv_accuracy": knn_search.best_score_, "test_accuracy": ocean_acc},
        "income": income_test,
        "income_cv": income_cv.drop(columns="_rmse").to_dict(orient="records"),
    }, f, indent=2, default=str)
print("Saved models/ocean_model.joblib, models/income_model.joblib, models/helper_metrics.json")

Saved models/ocean_model.joblib, models/income_model.joblib, models/helper_metrics.json
